# Notebook 3: Permutation Tests

---

**Author**: Laura Steel Pascual  
**Date**: February 2026  
**Project**: Decoding Anthropogenic Impact with eDNA and Supervised Machine Learning: Species‑Level Associations from Solomon Island Rivers

---

## Overview
This notebook validates optimised Random Forest model performance via **permutation testing** to confirm that classification accuracy exceeds chance levels. 

**Purpose**:
- Model validation via permutation testing.

**Methods** (for each of the 8 optimised Random Forest models):
1. Calculate observed AUC via 5-fold stratified cross-validation.
2. Generate null distribution of AUCs by randomly permuting labels 1000 times.
3. Compute p-value: proportion of permuted AUCs ≥ observed AUC.
4. Visualise null distributions with observed AUC markers.

**Statistical Framework:**
- Null hypothesis (H₀): Model performance is no better than random chance.
- Alternative hypothesis (H₁): Model discriminates reference vs impacted sites beyond chance.
- Significance levels: p < 0.05 (*), p < 0.005 (**), p < 0.001 (***)

**Paper Reference**: 2. Materials & Methods -> 2.6 Model Validation; 3. Results -> 3.2 Optimised Random Forest Performance and Statistical Validation

---

## 1. Setup

In [20]:
# Import libraries
import numpy as np
import pandas as pd
import sklearn
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
import warnings
warnings.filterwarnings('ignore')
import os
import pickle

# Create output directories
os.makedirs('results/permutation_tests', exist_ok=True)

print(f'NumPy version: {np.__version__}')
print(f'Pandas version: {pd.__version__}')
print(f'Scikit-learn version: {sklearn.__version__}')
print(f'Seaborn version: {sns.__version__}')

NumPy version: 1.24.4
Pandas version: 2.3.2
Scikit-learn version: 1.2.2
Seaborn version: 0.12.2


## 2. Configuration

In [6]:
# Buffer-threshold combinations
buffers = [100, 300, 500, 1000]
thresholds = [0.03125, 0.0625]

# Permutation test parameters
n_permutations = 1000
n_folds = 5
random_state = 42

cv_strategy = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=random_state)

## 3. Load Models and Data

Load the 8 optimised Random Forest models from Notebook 2 and their corresponding preprocessed datasets from Notebook 1.

In [9]:
from sklearn.metrics import accuracy_score

# Storage
models = {}
datasets = {}

print('Loading models and data...\n')

for buf in buffers:
    for thresh in thresholds:
        thresh_str = str(thresh).replace('.', '')
        key = f'{buf}m_t{thresh_str}'
        
        # Load data
        with open(f'results/preprocessed/data_{key}.pkl', 'rb') as f:
            datasets[key] = pickle.load(f)
        
        # Load model
        with open(f'results/models/rf_optimised_{key}.pkl', 'rb') as f:
            models[key] = pickle.load(f)
        
        n_ref = (datasets[key]['y'] == 0).sum()
        n_imp = (datasets[key]['y'] == 1).sum()
        print(f'{key}: {n_ref} reference, {n_imp} impacted')

print(f'\nLoaded {len(models)} models')

Loading models and data...

100m_t003125: 26 reference, 21 impacted
100m_t00625: 27 reference, 20 impacted
300m_t003125: 22 reference, 25 impacted
300m_t00625: 25 reference, 22 impacted
500m_t003125: 16 reference, 31 impacted
500m_t00625: 23 reference, 24 impacted
1000m_t003125: 18 reference, 29 impacted
1000m_t00625: 29 reference, 18 impacted

Loaded 8 models


## 4. Verify Model Performance

Calculate AUC and Accuracy for each model to confirm they match Notebook 2 results.

In [12]:
print('Verifying model performance (5-fold CV)...')
print('=' * 80)
print(f'{"Model":20s} {"AUC":>10s} {"Accuracy":>10s} {"Samples":>10s}')
print('=' * 80)

verification = []

for buf in buffers:
    for thresh in thresholds:
        thresh_str = str(thresh).replace('.', '')
        key = f'{buf}m_t{thresh_str}'
        
        X = datasets[key]['X_raw']
        y = datasets[key]['y']
        pipe = models[key]
        
        # Calculate metrics across folds
        auc_scores = []
        acc_scores = []
        
        for train_idx, test_idx in cv_strategy.split(X, y):
            # Use array indexing instead of iloc
            X_train, X_test = X[train_idx], X[test_idx]
            y_train, y_test = y[train_idx], y[test_idx]
            
            pipe.fit(X_train, y_train)
            
            y_pred = pipe.predict(X_test)
            y_proba = pipe.predict_proba(X_test)[:, 1]
            
            acc_scores.append(accuracy_score(y_test, y_pred))
            auc_scores.append(roc_auc_score(y_test, y_proba))
        
        mean_auc = np.mean(auc_scores)
        mean_acc = np.mean(acc_scores)
        
        verification.append({
            'key': key,
            'auc': mean_auc,
            'accuracy': mean_acc
        })
        
        print(f'{key:20s} {mean_auc:10.4f} {mean_acc:10.4f} {len(y):10d}')

print('=' * 80)
print('\nPerformance verification complete')

Verifying model performance (5-fold CV)...
Model                       AUC   Accuracy    Samples
100m_t003125             0.7077     0.5911         47
100m_t00625              0.7433     0.6756         47
300m_t003125             0.7900     0.6844         47
300m_t00625              0.7720     0.7422         47
500m_t003125             0.6571     0.6200         47
500m_t00625              0.7860     0.7467         47
1000m_t003125            0.8856     0.7867         47
1000m_t00625             0.9394     0.8044         47

Performance verification complete


## 5. Run Permutation Tests

For each model: run 1000 permutations with shuffled labels.

**Note:** This may take some time.

In [15]:
results = []
print(f'Running permutation tests ({n_permutations} rounds per model)...')
print('This may take some time\n')

for buf in buffers:
    for thresh in thresholds:
        thresh_str = str(thresh).replace('.', '')
        key = f'{buf}m_t{thresh_str}'
        
        print(f'\n{"=" * 80}')
        print(f'{key}')
        print("=" * 80)
        
        # Get data and model
        X = datasets[key]['X_raw']
        y = datasets[key]['y']
        pipe = models[key]
        
        # Get real AUC from verification
        real_auc = [v['auc'] for v in verification if v['key'] == key][0]
        print(f'Real AUC: {real_auc:.4f}')
        
        # Permutation test
        print(f'Running permutation test...')
        permuted_aucs = []
        
        for i in range(n_permutations):
            if (i + 1) % 100 == 0 or i == 0:
                print(f'  Permutation {i + 1}/{n_permutations}...')
            
            # Shuffle labels - CHANGED: Use numpy instead of pandas
            np.random.seed(random_state + i)
            y_permuted = np.random.permutation(y)
            perm_auc_scores = []
            
            for train_idx, test_idx in cv_strategy.split(X, y_permuted):
                # CHANGED: Use array indexing [idx] instead of .iloc[idx]
                X_train, X_test = X[train_idx], X[test_idx]
                y_train, y_test = y_permuted[train_idx], y_permuted[test_idx]
                
                pipe.fit(X_train, y_train)
                y_proba = pipe.predict_proba(X_test)[:, 1]
                perm_auc_scores.append(roc_auc_score(y_test, y_proba))
            
            permuted_aucs.append(np.mean(perm_auc_scores))
        
        # Calculate p-value
        p_value = np.mean([auc >= real_auc for auc in permuted_aucs])
        mean_permuted_auc = np.mean(permuted_aucs)
        
        # Determine significance
        if p_value < 0.001:
            sig = '***'
        elif p_value < 0.005:
            sig = '**'
        elif p_value < 0.05:
            sig = '*'
        else:
            sig = ''
        
        print(f'\nPermutation Test Results:')
        print(f'Real Mean AUC:     {real_auc:.4f}')
        print(f'Mean Permuted AUC: {mean_permuted_auc:.4f}')
        print(f'P-value:           {p_value:.4f} {sig}')
        
        # Store results
        results.append({
            'Model': f'RFChp_{key}_5f',
            'Buffer (m)': buf,
            'LUI Threshold': thresh,
            'Real Mean AUC': real_auc,
            'Mean Permuted AUC': mean_permuted_auc,
            'P-value': p_value,
            'Sig.': sig
        })
        
        # Plot distribution
        plt.figure(figsize=(8, 6))
        sns.histplot(permuted_aucs, bins=30, kde=True, color='gray')
        plt.axvline(real_auc, color='red', linestyle='--', label=f'Real AUC = {real_auc:.4f}')
        plt.xlabel('Permuted AUC')
        plt.ylabel('Frequency')
        plt.title(f'Permutation Test: {buf}m Buffer, LUI {thresh}\np-value = {p_value:.4f} {sig}')
        plt.legend()
        plt.savefig(f'results/permutation_tests/perm_dist_{key}.png', dpi=300, bbox_inches='tight')
        plt.close()
        print(f'Saved: perm_dist_{key}.png')

print(f'\n{"=" * 80}')
print('Permutation testing complete')

Running permutation tests (1000 rounds per model)...
This may take some time


100m_t003125
Real AUC: 0.7077
Running permutation test...
  Permutation 1/1000...
  Permutation 100/1000...
  Permutation 200/1000...
  Permutation 300/1000...
  Permutation 400/1000...
  Permutation 500/1000...
  Permutation 600/1000...
  Permutation 700/1000...
  Permutation 800/1000...
  Permutation 900/1000...
  Permutation 1000/1000...

Permutation Test Results:
Real Mean AUC:     0.7077
Mean Permuted AUC: 0.5225
P-value:           0.0430 *
Saved: perm_dist_100m_t003125.png

100m_t00625
Real AUC: 0.7433
Running permutation test...
  Permutation 1/1000...
  Permutation 100/1000...
  Permutation 200/1000...
  Permutation 300/1000...
  Permutation 400/1000...
  Permutation 500/1000...
  Permutation 600/1000...
  Permutation 700/1000...
  Permutation 800/1000...
  Permutation 900/1000...
  Permutation 1000/1000...

Permutation Test Results:
Real Mean AUC:     0.7433
Mean Permuted AUC: 0.5121
P-value:       

## 6. Save Results

In [22]:
# Create summary table
summary_df = pd.DataFrame(results)

# Save Excel
summary_df.to_excel('results/permutation_tests/permutation_summary.xlsx', index=False, sheet_name='Permutation Tests')
print('Saved: permutation_summary.xlsx')

# Display
print('\nPermutation Test Summary:')
print(summary_df.to_string(index=False))

Saved: permutation_summary.xlsx

Permutation Test Summary:
                 Model  Buffer (m)  LUI Threshold  Real Mean AUC  Mean Permuted AUC  P-value Sig.
 RFChp_100m_t003125_5f         100        0.03125       0.707667           0.522546    0.043    *
  RFChp_100m_t00625_5f         100        0.06250       0.743333           0.512090    0.024    *
 RFChp_300m_t003125_5f         300        0.03125       0.790000           0.523068    0.004   **
  RFChp_300m_t00625_5f         300        0.06250       0.772000           0.526352    0.010    *
 RFChp_500m_t003125_5f         500        0.03125       0.657143           0.502861    0.103     
  RFChp_500m_t00625_5f         500        0.06250       0.786000           0.510008    0.005    *
RFChp_1000m_t003125_5f        1000        0.03125       0.885556           0.485406    0.000  ***
 RFChp_1000m_t00625_5f        1000        0.06250       0.939444           0.484064    0.000  ***


## 7. Summary

**Outputs** (`results/permutation_tests/`):

- `permutation_summary.xlsx`
- `perm_dist_100m_t003125.png` through `perm_dist_1000m_t00625.png`

7 of 8 models achieve statistical significance (p < 0.05).

## Next Steps 
- **Notebook 4** (`4_Feature_Importance.ipynb`): Feature importance and SHAP analysis for the 7 statistically significant models (p < 0.05).